# 09 · Bring your own universe: a complete, honest research cycle

**Who this is for:** anyone evaluating First-Mover Flow on their own data. The cycle is the same one the paper followed:

1. **Plan with power.** Given your universe size and history, what is the smallest effect you could detect?
2. **Split** history into a development period and a confirmation period. Do all exploration on development only.
3. **Freeze and pre-register** the specification (a SHA-256 fingerprint) before touching the confirmation period.
4. **Confirm** with the full battery: performance, signal ICs, comparables, placebos, leave-one-out, Bayesian posterior.
5. **Report** what came out, whatever it is.

A synthetic universe stands in for "your data" (40 stocks, 7 years, a sector map), so the notebook runs anywhere. Replace the two `generate(...)` lines with `load_prices` / `load_ratings` on your files.

In [1]:
# fmflow-setup: runs as-is everywhere. In an installed environment it only locates the repository; on Google Colab,
# Binder or a Jupyter without the package, it downloads the repository once and installs it (about a minute).
import importlib, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get("FMFLOW_REPO_URL", "https://github.com/bahakh19/first-mover-flow")   # set it to use a fork


def _run(cmd: list, what: str, hint: str = "") -> None:
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{what} failed:\n{(r.stderr or r.stdout).strip()[-1500:]}\n{hint}")


_here = Path.cwd()
_root = next((p for p in (_here, *_here.parents, _here / "first-mover-flow") if (p / "src" / "fmflow").is_dir()), None)
if _root is None:                                                  # notebooks need the repository's data/ folder
    _root = (_here / "first-mover-flow").resolve()
    print(f"Downloading {REPO_URL} ...")
    if shutil.which("git"):
        _run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(_root)], f"git clone {REPO_URL}",
             "Is the repository public? For a fork or a local copy, set the environment variable FMFLOW_REPO_URL.")
    else:                                                          # no git: the GitHub zip archive of the main branch
        import io, urllib.request, zipfile
        with urllib.request.urlopen(REPO_URL.rstrip("/") + "/archive/refs/heads/main.zip") as _r:
            zipfile.ZipFile(io.BytesIO(_r.read())).extractall(_here)
        (_here / (REPO_URL.rstrip("/").split("/")[-1] + "-main")).rename(_root)
if importlib.util.find_spec("fmflow") is None:
    print("Installing fmflow and its dependencies (about a minute) ...")
    _target = f"{_root}[plot]"
    if importlib.util.find_spec("pip"):
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", _target], "pip install")
    elif shutil.which("uv"):                                       # uv-created environments have no pip
        _run(["uv", "pip", "install", "-q", "--python", sys.executable, "-e", _target], "uv pip install")
    else:
        raise RuntimeError("this kernel has neither pip nor uv: install the package from a terminal (INSTALL.md)")
    sys.path.insert(0, str(_root / "src"))
    importlib.invalidate_caches()
os.chdir(_root / "notebooks")
print("fmflow", importlib.import_module("fmflow").__version__, "| repository:", _root)

fmflow 2.2.4 | repository: /home/claude/work/v22/repo


In [2]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tempfile
from fmflow import StrategyConfig, run_strategy
from fmflow.synthetic import SyntheticConfig, generate, describe_dataset
from fmflow.analytics.power import mde_ic, ic_power
pd.set_option("display.float_format", "{:.3f}".format)
prices, ratings = generate(SyntheticConfig(n_stocks=40, n_sessions=252 * 7, n_sectors=4, seed=23, strength=0.3,
                                           analysts_per_stock=(6, 30), analyst_pool_ratio=0.3))
sectors = {f"stock_{i + 1}": f"sector_{i % 4}" for i in range(40)}        # your GICS / ICB map (optional, see below)
display(describe_dataset(prices, ratings).to_frame("your universe"))

,your universe
stocks,40.000
sessions,1764.000
avg pairwise corr,0.340
ann vol min,0.273
ann vol max,0.555
ratings,13431.000
analysts,213.000
analysts per stock (median),18.500
stocks per analyst (median),3.000
median days between an analyst's updates,85.000


## 1. Power before anything else

In [3]:
n = prices["stock"].nunique(); years_conf = 3
print(f"Minimum detectable mean IC with {n} stocks over a {years_conf}-year confirmation period: {mde_ic(n, years_conf * 12):.3f}")
print("Power for an IC of 0.05 / 0.10:", round(ic_power(0.05, n, years_conf * 12), 2), "/", round(ic_power(0.10, n, years_conf * 12), 2))

Minimum detectable mean IC with 40 stocks over a 3-year confirmation period: 0.066
Power for an IC of 0.05 / 0.10: 0.59 / 0.98


## 2. Development period: explore freely, but only here

In [4]:
dates = sorted(prices["date"].unique())
split = dates[int(len(dates) * 4 / 7)]                                        # first 4 years = development
dev_p, dev_r = prices[prices["date"] <= split], ratings[ratings["date"] <= split]
base = StrategyConfig.fmf_n_scalable().with_(allocator="EW")
from fmflow.robustness import multiverse, signal_ic_table
dev_w = {"dev": slice(dates[252], split)}
tab, _ = multiverse(dev_p, dev_r, base, {"tranche_period": [21, 42, 63], "lmh_window": [63, 126, 252], "legs": [0.3, 0.4]},
                    dev_w, mode="oat")
display(tab[["Sharpe dev", "Calmar dev"]])

,Sharpe dev,Calmar dev
spec,,
base,1.449,0.740
tranche_period=21,1.614,0.827
tranche_period=42,1.449,0.740
tranche_period=63,1.628,0.893
lmh_window=63,1.462,0.826
lmh_window=126,1.449,0.740
lmh_window=252,1.445,0.832
legs=0.3,1.507,0.746
legs=0.4,1.449,0.740


## 3. Freeze and pre-register

Development favoured the defaults, so the specification is frozen as is. The registration file states what will be tested, and how, **before** the confirmation data are touched.

In [5]:
from fmflow.prereg import register, verify
frozen = base.with_(name="FMF-N (my universe)")
reg_path = Path(tempfile.mkdtemp()) / "my_prereg.json"
doc = register({"title": "FMF-N on my universe: confirmation test", "frozen": frozen.to_dict(),
                "confirmation_window": [split, dates[-1]], "primary_test": "net Sharpe > 0 and LMH IC21 > 0 (one-sided 5%)",
                "decision_rule": "No parameter is changed after this file is hashed."}, reg_path)
print("registered", doc["sha256_16"], doc["registered_utc"])

registered c658d6304856d075 2026-10-29T03:48:39Z


## 4. Confirmation: the full battery, once

In [6]:
verify(reg_path)                                                             # fails if the file was edited
out = run_strategy(prices, ratings, frozen)       # pass sectors=sectors here ONLY if development used them too
cw = slice(dates[dates.index(split) + 1], dates[-1])
from fmflow.analytics.metrics import perf_stats, capm
from fmflow.analytics.bayes import bayesian_sharpe
from fmflow.robustness import strategy_placebo
n_ = out.net_returns.loc[cw]
print({k: round(v, 3) for k, v in perf_stats(n_)[["Sharpe", "Calmar", "Max drawdown"]].items()},
      {k: round(v, 3) for k, v in capm(n_, out.returns.mean(axis=1).loc[cw]).items()})
display(signal_ic_table(out, 21, cw).loc[["leader_minus_herd", "composite"]])
display(strategy_placebo(out, {"confirmation": cw}, n_iid=100, n_perm=100, shift_step=10))
print("posterior Sharpe (sceptical prior):", {k: round(v, 3) for k, v in bayesian_sharpe(n_, prior_sr_sd=1.0).summary().items()})

{'Sharpe': 1.43, 'Calmar': 1.469, 'Max drawdown': -0.061} {'beta': 0.028, 'alpha_ann': 0.085, 'alpha_t': 2.35}


,IC,t_phase,p (one-sided),hit rate,coverage,days
leader_minus_herd,0.056,1.758,0.044,0.602,0.960,734.000
composite,0.065,2.240,0.016,0.650,0.968,734.000


draws  observed  null_median  null_95     p
confirmation iid         100.000     1.430       -0.528    0.778 0.020
             permutation 100.000     1.430        0.121    1.254 0.040
             time_shift  152.000     1.430        0.816    4.804 0.190

posterior Sharpe (sceptical prior): {'mean': 1.108, 'median': 1.117, 'sd': 0.507, 'hdi90_low': 0.277, 'hdi90_high': 1.939, 'P(>0)': 0.985, 'P(>0.5)': 0.883, 'P(>1)': 0.589}


**Reading the outcome.** A confirmation can fail even when the effect is real: with this universe size the power to
detect an IC of 0.05 is only about 60% (step 1). With a different simulated history (`seed=21`), the same true effect
gives a development Sharpe of 1.7 and a confirmation Sharpe of 0.3. Never "fix" a failed confirmation by changing the
specification: register a new test on new data.

## 5. Report

Write the results to the registration's companion file and publish both, whatever the outcome. The command-line equivalent of the battery is:

```bash
fmflow register --config my_config.json --title "..." --out configs/my_prereg.json
fmflow evaluate --prices P.csv --ratings R.csv --config my_config.json --placebo --loo --out results/confirmation/
fmflow infer    --prices P.csv --ratings R.csv --config my_config.json
```